# VietHandOCR Part 5: Evaluation & Inference

Welcome to **Part 5** of the VietHandOCR pipeline.
- **Previous Notebook**: [Part 4: VietOCR Model Training](./04_VietOCR_Training.ipynb)

## Introduction
We load our fine-tuned weights and evaluate against the test set, computing Character Error Rate (CER), Word Error Rate (WER), Exact Match, and BLEU. Finally, we export the model to **ONNX** format for accelerated, low-latency inference.



In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
!pip install -q jiwer nltk
import jiwer
import nltk
nltk.download('punkt')
def calculate_metrics(predictions, targets):
    '''Calculates CER, WER, Exact Match, and BLEU.'''
    cer = jiwer.cer(targets, predictions)
    wer = jiwer.wer(targets, predictions)
    exact_match = sum(1 for p, t in zip(predictions, targets) if p == t) / len(targets)
    
    bleu_scores = []
    for p, t in zip(predictions, targets):
        reference = [nltk.word_tokenize(t.lower())]
        candidate = nltk.word_tokenize(p.lower())
        bleu = nltk.translate.bleu_score.sentence_bleu(reference, candidate, weights=(0.5, 0.5))
        bleu_scores.append(bleu)
    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
    
    return {"CER": cer, "WER": wer, "Exact Match": exact_match, "BLEU": avg_bleu}
def visualize_failures(images, predictions, targets):
    '''Plots images where predictions != targets to analyze failure modes.'''
    pass


## Inference Optimization (ONNX Export)
To ensure our model runs efficiently in a production setting, we convert the PyTorch model to ONNX format. This allows for potential TensorRT optimization or int8 quantization.

In [ ]:
def export_to_onnx(model, dummy_input, output_path="vietocr_model.onnx"):
    '''Exports the fine-tuned PyTorch model to ONNX format.'''
    # NOTE: VietOCR models are sequence-to-sequence. 
    # To properly export to ONNX, the CNN encoder and Transformer decoder must be exported separately.
    # Below is a skeletal export for the CNN feature extractor only.
    # For full autoregressive inference export, see official VietOCR documentation.
    
    cnn_model = model.cnn
    cnn_model.eval()
    
    torch.onnx.export(
        cnn_model, 
        dummy_input, 
        output_path, 
        export_params=True, 
        opset_version=14, 
        do_constant_folding=True,
        input_names=['input_images'], 
        output_names=['cnn_features'],
        dynamic_axes={'input_images': {0: 'batch_size', 3: 'width'},
                      'cnn_features': {0: 'batch_size', 1: 'sequence_length'}}
    )
    print(f"CNN feature extractor successfully exported to {output_path}")
# cnn_dummy = torch.randn(1, 3, 32, 256).to('cuda' if torch.cuda.is_available() else 'cpu')
# export_to_onnx(model, dummy_input=cnn_dummy)
